# 04 · Recover the leaf groups of images without a verified leaf id (D-010)

**Why.** Verified leaves have a median of 4 photos each, but 97% of the file-name ("fallback") leaf ids contain one
image: for 24% of the dataset, other photos of the same leaf are not grouped and can sit in train and test.

**How.** MobileNetV2 (ImageNet) embeddings → k nearest neighbours inside each class → the similarity cut-off is
**calibrated on verified leaves**, where the true answer is known → pairs above it that involve a fallback image are
grouped before splitting, like the near-identical pairs of D-006.

**Run it in two parts.** Select a **T4 GPU** runtime. Click the code cell under *"8 · Decision"* and use
**Runtime → Run before**, then send screenshots of sections 2, 5, 6 and 7. After the cut-off is chosen, set it in
section 8 and run that cell and everything below (**Runtime → Run cell and below**). About 20–30 minutes.

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # the Colab copy is disposable: always match GitHub exactly
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
head = !git -C {REPO_DIR} rev-parse HEAD
remote = !git -C {REPO_DIR} rev-parse origin/main
assert head == remote, "repo is not at the latest commit: run this cell again"
!git -C {REPO_DIR} log -1 --oneline
!nvidia-smi -L

## 2 · Images, and their sizes (plan step 2.2)

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
from PIL import Image

from plantdx.data.images import download_archive, extract_variant, image_path

VARIANT, DATA_ROOT = "color", "/content/data"
print("new files extracted:", extract_variant(download_archive(), DATA_ROOT, VARIANT))

parts = [pd.read_csv(f"splits/{s}.csv").assign(split=s) for s in ("train", "val", "test")]
parts.append(pd.read_csv("splits/dropped_train_overlap.csv").assign(split="removed"))
universe = pd.concat(parts, ignore_index=True)

sizes = Counter()
for rel in universe["path"]:
    with Image.open(image_path(DATA_ROOT, VARIANT, rel)) as im:   # reads the header only
        sizes[f"{im.size[0]}x{im.size[1]} {im.mode} {im.format}"] += 1
image_sizes = {"images": len(universe), "kinds": dict(sizes.most_common())}
Path("reports").mkdir(exist_ok=True)
Path("reports/image_sizes.json").write_text(json.dumps(image_sizes, indent=2) + "\n")
print(image_sizes)

## 3 · Embeddings (cached on Drive)

Two variants: plain, and averaged over the 8 rotations/flips (`d4`), so that the pose of the leaf matters less.

In [ ]:
import numpy as np

from plantdx.data.pipeline import make_dataset
from plantdx.data.sameleaf import build_embedder, embed, load_embeddings, save_embeddings
from plantdx.paths import artifacts_dir

emb_dir = artifacts_dir() / "embeddings"; emb_dir.mkdir(parents=True, exist_ok=True)
paths = universe["path"].tolist()
BATCH = 128
model = build_embedder(224)
embeddings = {}
for tta in ("none", "d4"):
    f = emb_dir / f"mobilenetv2_imagenet_{tta}.npz"
    e = load_embeddings(f, paths)
    if e is None:
        ds = make_dataset(universe, DATA_ROOT, VARIANT, image_size=224, batch_size=BATCH, num_classes=38)
        e = embed(ds, model, tta=tta, total_batches=int(np.ceil(len(paths) / BATCH)))
        save_embeddings(f, e, paths)
    embeddings[tta] = e
    print(tta, e.shape)

## 4 · Nearest neighbours inside each class

In [ ]:
from plantdx.data.sameleaf import annotate, knn_pairs

K = 10
pairs = {tta: annotate(knn_pairs(universe, e, k=K), universe) for tta, e in embeddings.items()}
for tta, p in pairs.items():
    print(tta, f"{len(p):,} pairs | known (both verified): {p.known.sum():,} | involving fallback: {p.involves_fallback.sum():,}")

## 5 · Calibration on verified leaves  ← **screenshot**

Precision: of the neighbour pairs above the cut-off whose leaves are known, how many are the same leaf.
Recall: how many of all same-leaf pairs among verified images are found. The suggested cut-off is the lowest one
where precision stays ≥ 0.95 for every higher cut-off.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

from plantdx.data.sameleaf import calibration_curve, pick_threshold

FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
curves, picks = {}, {}
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for tta, p in pairs.items():
    for mutual in (False, True):
        name = f"{tta}{' mutual' if mutual else ''}"
        c = calibration_curve(p, universe, mutual_only=mutual)
        curves[name], picks[name] = c, pick_threshold(c, 0.95)
        ax.plot(c["recall"], c["precision"], marker=".", label=name)
ax.axhline(0.95, color="k", ls="--", lw=0.8)
ax.set_xlabel("recall of same-leaf pairs (verified leaves)"); ax.set_ylabel("precision")
ax.set_title("Same-leaf detection calibrated on verified leaves"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "sameleaf_calibration.png", dpi=130); plt.close(fig)
display(IPImage(str(FIG / "sameleaf_calibration.png")))

summary_rows = []
for name, c in curves.items():
    t = picks[name]
    row = c[c.threshold == t].iloc[0].to_dict() if t is not None else {"threshold": None}
    summary_rows.append({"variant": name, **row})
calib = pd.DataFrame(summary_rows)
calib.to_csv("reports/sameleaf_calibration.csv", index=False)
pd.concat([c.assign(variant=n) for n, c in curves.items()]).to_csv("reports/sameleaf_curves.csv", index=False)
display(calib)
best = calib.dropna(subset=["recall"]).sort_values("recall", ascending=False).iloc[0]
print("best variant at precision >= 0.95:", best["variant"], "| cut-off", best["threshold"], "| recall", best["recall"])

## 6 · Look at pairs with a fallback image near the cut-off  ← **screenshot**

For the best variant: random pairs that involve a fallback image, in bands around the suggested cut-off. Are they the
same leaf?

In [ ]:
from plantdx.viz import plot_pairs

BEST = best["variant"]; TTA = BEST.split()[0]; MUTUAL = "mutual" in BEST; T0 = float(best["threshold"])
fb = pairs[TTA][pairs[TTA].involves_fallback & (pairs[TTA].mutual | (not MUTUAL))]
abbr = {"train": "tr", "val": "va", "test": "te", "removed": "rm"}
for lo, hi in ((T0 - 0.06, T0 - 0.03), (T0 - 0.03, T0), (T0, T0 + 0.03), (T0 + 0.03, 1.01)):
    band = fb[(fb.sim >= lo) & (fb.sim < hi)]
    print(f"band {lo:.2f}–{min(hi, 1):.2f}: {len(band):,} pairs")
    if band.empty:
        continue
    pick = band.sample(min(7, len(band)), random_state=0)
    labels = [f"s={s:.3f} {abbr[a]}/{abbr[b]}" for s, a, b in zip(pick["sim"], pick["split_a"], pick["split_b"])]
    display(IPImage(str(plot_pairs(pick, DATA_ROOT, VARIANT, f"/content/band_{lo:.2f}.jpg", labels=labels,
                                   title=f"{BEST}: similarity {lo:.2f}–{min(hi, 1):.2f} (pairs with a fallback image)"))))

## 7 · What each cut-off would do to the splits  ← **screenshot**

Together with the near-identical pairs already in `splits/duplicate_pairs.csv`. A good result: fallback groups get a
size distribution like verified leaves (median about 4), the largest group stays reasonable, and the number of train
images removed is acceptable.

In [ ]:
from plantdx.data.sameleaf import group_size_stats
from plantdx.data.split import load_group_pairs, merge_duplicate_groups

existing = load_group_pairs("splits")
existing = existing[existing.source == "duplicate_pairs"] if existing is not None else None
rows = []
for t in np.round(np.arange(T0 - 0.06, T0 + 0.061, 0.02), 2):
    p = fb[fb.sim >= t]
    allp = pd.concat([existing[["path_a", "path_b"]], p[["path_a", "path_b"]]]) if existing is not None else p
    g = merge_duplicate_groups(universe, allp)
    u = universe.assign(group=g.values)
    test_groups = set(u.loc[u.split == "test", "group"])
    trval = u[u.split.isin(["train", "val"])]
    stats = group_size_stats(u, u["group"], u["leaf_source"] == "fallback")
    rows.append({"cut_off": t, "pairs": len(p), "cross_split": int(p.cross_split.sum()),
                 "train_val_to_remove": int(trval.group.isin(test_groups).sum()),
                 "largest_group": int(u.groupby("group").size().max()),
                 "fallback_group_median": stats["median"], "fallback_single_share": stats["single_image_share"]})
cost = pd.DataFrame(rows)
cost.to_csv("reports/sameleaf_threshold_cost.csv", index=False)
display(cost)

## 8 · Decision

Set the variant and cut-off agreed after looking at sections 5–7, then **Runtime → Run cell and below**.

In [ ]:
SAMELEAF_VARIANT = BEST          # e.g. "d4 mutual"
SAMELEAF_THRESHOLD = T0          # e.g. 0.80

tta, mutual = SAMELEAF_VARIANT.split()[0], "mutual" in SAMELEAF_VARIANT
p = pairs[tta]
chosen = p[p.involves_fallback & (p.sim >= SAMELEAF_THRESHOLD) & (p.mutual | (not mutual))]
cols = ["path_a", "path_b", "label_index", "sim", "mutual"]
chosen.sort_values(["label_index", "path_a", "path_b"])[cols].to_csv("splits/sameleaf_pairs.csv", index=False)
c = curves[SAMELEAF_VARIANT]
at = c.loc[(c.threshold - SAMELEAF_THRESHOLD).abs().idxmin()]
meta = {
    "method": "MobileNetV2 (ImageNet) global-average embeddings, cosine k-NN inside each class (plantdx.data.sameleaf)",
    "variant": SAMELEAF_VARIANT, "k": K, "threshold": SAMELEAF_THRESHOLD,
    "calibration_on_verified_leaves": {"precision": float(at.precision), "recall": float(at.recall)},
    "pairs": int(len(chosen)), "cross_split_pairs_before": int(chosen.cross_split.sum()),
}
Path("splits/sameleaf_meta.json").write_text(json.dumps(meta, indent=2) + "\n")
print(json.dumps(meta, indent=2))

## 9 · Rebuild the splits and refresh the data card

In [ ]:
!python -m plantdx.data.prepare
!python -m plantdx.data.card

## 10 · Verify

In [ ]:
from plantdx.data.split import assert_no_leakage, cross_split_pairs
from plantdx.viz import plot_class_counts, plot_fallback_share, plot_images_per_leaf

new = pd.concat([pd.read_csv(f"splits/{s}.csv").assign(split=s) for s in ("train", "val", "test")], ignore_index=True)
assert_no_leakage(new, load_group_pairs("splits"))
assert set(new.loc[new.split == "test", "path"]) == set(universe.loc[universe.split == "test", "path"])
print("pairs still across splits:", len(cross_split_pairs(new, chosen)))
print("fallback groups:", group_size_stats(new, new["group_id"], new["leaf_source"] == "fallback"))
classes = json.load(open("splits/classes.json"))
plot_class_counts(new, classes, FIG / "eda_class_counts.png")
plot_fallback_share(new, classes, FIG / "eda_fallback_share.png")
plot_images_per_leaf(new, FIG / "eda_images_per_leaf.png")
print(json.dumps({k: v for k, v in json.load(open("splits/summary.json")).items() if k != "settings"}, indent=2))

## 11 · Download

In [ ]:
from google.colab import files

!rm -f /content/step2_sameleaf.zip && zip -qr /content/step2_sameleaf.zip splits reports docs/data_card.md
files.download("/content/step2_sameleaf.zip")